Theorem1のclaim(i)から、uがvの親でなくて、vの親がC(vの子孫とvとuを除くノード)に入っていないならば、τはゼロ。つまり恒等式が成り立つ？
逆にずれが出るのはclaim2。


Y1 = e1
Y2 = b * Y1 + e2
とする。このときbはE(Y1^(K-1) * Y2)/E(Y1^K)で出る。Kが1より大きいなら成り立つ。それで、K次のときと2次の時の係数が一致することを利用して恒等式を立てる。ただしY1とY2を入れ替えたり、誤差が非ガウスだとこの恒等式は成り立たない。



adaptive_lassoを使うのは、枝刈り、親を探すときに係数が大きいものを残すとか、ブートストラップして係数が生き残る確率が高いものを選ぶとか？

In [3]:
import itertools
import numpy as np
from sklearn.preprocessing import scale
from sklearn.utils import check_array, check_scalar
from scipy.linalg import pinvh

from lingam import DirectLiNGAM

np.random.seed(0)

class HighDimDirectLiNGAM(DirectLiNGAM):

    def __init__(self, K=4, J=3, **kwargs):
        super().__init__(**kwargs)

        self._K = check_scalar(K, "K", int, min_val=2,include_boundaries="neither")
        self._J = check_scalar(J, "J", int, min_val=1)

    def fit(self, X):
        """Fit the model to X.

        Parameters
        ----------
        X : array-like, shape (n_samples, n_features)
            Training data, where ``n_samples`` is the number of samples
            and ``n_features`` is the number of features.

        Returns
        -------
        self : object
            Returns the instance itself.
        """
        # Check parameters
        Y = check_array(X, copy=True)
        # centered
        
        n_features = Y.shape[1]

        # Causal discovery
        theta = [] 
        psi = set(range(n_features))
        # C[z][v]
        C = {}
        
        # 書き換えることのない固定値
        self._Y = Y
        self._alpha = 0.5
        self._sigma = np.cov(self._Y.T, bias=True)
        
        # XXX: キャッシュを持たせる
        self._inv_cache = {}
        self._Yvi_C_cache = {}
        
        g = []
        
        for z in range(n_features):
            Cz = {}
            stats = []
            
            if z == 0:
                # XXX: z=0ではどのみち前ステップを元にした候補がないので g(0) = 0?
                gz = 0
            else:
                r = theta[-1]
                gz = max([*g, self._alpha * self._T(r, C[z - 1][r], psi)])
            
            for v in psi:
                Cv = self._possible_parents(z, v, C, theta, psi, gz)
                stat = self._T(v, Cv, psi - set([v]))
                
                Cz[v] = Cv
                stats.append((stat, v))
            _, r = min(stats)
            theta.append(r)
            psi.remove(r)
            C[z] = Cz
            g.append(gz)
            
            # XXX: len(psi) == 1だと_T()のV2が空になる。最後は自動的に決まるので終了処理。
            if len(psi) == 1:
                theta.append(psi.pop())
                break

        # XXX: prune?
        
        self._causal_order = theta
        return self._estimate_adjacency_matrix(X, prior_knowledge=self._Aknw)
        #self._adjacency_matrix = np.zeros((len(Y.T), len(Y.T)))
        #return self

    # old
    def _tau(self, v, u, C):
        if C is None:
            Yvi_C = self._Y[:, v]
        else:
            if (C, v) not in self._Yvi_C_cache:
                if C not in self._inv_cache:
                    subcov_inv = np.linalg.pinv(self._sigma[C, :][:, C])
                else:
                    subcov_inv = self._inv_cache[C]

                Bvc = subcov_inv @ self._sigma[C, v]    
                Yvi_C = self._Y[:, v] - self._Y[:, C] @ Bvc
                
                if C not in self._inv_cache:
                    self._inv_cache[C] = subcov_inv
            else:
                Yvi_C = self._Yvi_C_cache[(C, v)]

        # XXX: 期待値を平均値で計算
        #tau = np.mean(Yvi_C ** (self._K - 1) * self._Y[:, u]) * np.mean(Yvi_C ** 2) \
        #                        - np.mean(Yvi_C ** self._K) * np.mean(Yvi_C * self._Y[:, u])
        #tau = ((Yvi_C ** (self._K - 1) @ self._Y[:, u])  / len(self._Y)) * ((Yvi_C @ Yvi_C) / len(self._Y)) \
        #                        - np.mean(Yvi_C ** self._K) * ((Yvi_C @ self._Y[:, u]) / len(self._Y))

        # 期待値をmeanで考えているけど、meanを使わずに総和でよくないか？みんなサンプル数で割らないなら同じだし。
        # あるいはログとか。
        pow_y = Yvi_C ** (self._K - 1)
        tau = (pow_y @ self._Y[:, u]) * (Yvi_C @ Yvi_C) - (pow_y @ Yvi_C) * (Yvi_C @ self._Y[:, u])
        return tau

    # new batched
    def _tau(self, v, u, C):
        if C is None:
            Yvi_C = self._Y[:, v]            
        else:
            subcov_inv = np.linalg.pinv(self._sigma[C, :][:, C])
            Bvc = subcov_inv @ self._sigma[C, v]    
            Yvi_C = self._Y[:, v] - self._Y[:, C] @ Bvc
        tau = np.mean(Yvi_C ** (self._K - 1) * self._Y[:, u]) * np.mean(Yvi_C ** 2) - np.mean(Yvi_C ** self._K) * np.mean(Yvi_C * self._Y[:, u])
        return tau
    
    def _possible_parents(self, z, v, C, theta, psi, gz):
        # XXX: 前ステップのルートrを参照するが最初はないので、親候補なしとした。しきいはなし(0)でいいだろう
        if len(theta) == 0:
            return set()

        Cv = set()
        for p in C[z - 1][v]:
            # Dv
            Dv = set()
            for d in range(z):
                # XXX: Dvに空集合は入るのか？
                for i in range(1, self._J + 1):
                    comb = itertools.combinations(C[d][v] - set([p]), i)
                    Dv |= set(list(comb))

            # 判定
            taus = []
            for C_ in Dv:
                tau = self._tau(v, p, C_)
                taus.append(tau)
                
            # XXX: 条件を満たさなかったり候補なしの時はスキップ
            if len(taus) == 0 or min(taus) <= gz:
                continue
            
            Cv |= set([p])
            
        Cv |= set(theta)
        return Cv

    def _T(self, v, V1, V2, is_T1=True):
        # XXX: V1が空だと空集合の要素を探すことになってしまう。
        # XXX: そこは無視してCは空としてtauの計算はできる。
        # XXX: tauの計算時のYui.Cについて、Cが空のとき残差はデータと同一。
        if len(V1) == 0:
            # XXX: V1Jのループはない。V1Jが空集合なので消してみた。
            taus = []
            for u in V2:
                tau = self._tau(v, u, None)
                taus.append((abs(tau ** (self._K)), u))
            tau, C = max(taus)
            return tau
        
        if self._J <= len(V1):
            V1J = list(itertools.combinations(V1, self._J))
        else:
            V1J = [tuple(V1)]
        
        if is_T1:
            max_taus = []
            for C in V1J:                
                taus = []
                for u in V2:
                    tau = self._tau(v, u, C)
                    tau = abs(tau ** self._K)
                    taus.append(tau)
                max_tau = max(taus)
                max_taus.append((max_tau, C))
            tau, C = min(max_taus)
        else:
            min_taus = []
            for u in V2:
                taus = []
                for C in V1J:
                    tau = self._tau(v, u, C)
                    tau = abs(tau ** self._K)
                    taus.append(tau)
                min_tau = min(taus)
                min_taus.append((min_tau, C))
            tau, C = max(min_taus)
        
        return tau

In [4]:
from datetime import datetime
import pickle
import os
from multiprocessing import Pool

import numpy as np
import pandas as pd
import lingam
from scipy.stats import kendalltau

import matplotlib.pyplot as plt

np.random.seed(0)

# 通常のランダム生成
def make_dag_and_errors(n_nodes, sample_size, J=3, weight=0.5):
    dag = np.zeros((n_nodes, n_nodes))
    errors = None
    
    # For each node v
    for v in range(n_nodes):        
        #  select the number of parents
        num_pa = np.random.choice(np.arange(min(n_nodes, J)))

        if v - 1 >= 0:
            # We include edge (v − 1, v) to ensure that the ordering is unique
            dag[v, v - 1] = np.random.choice([-1, 1]) * np.random.uniform(weight, 1)

        # The remaining parents are selected uniformly from [v − 2]
        candidates = np.arange(v - 1)
        if len(candidates) == 0:
            continue

        # num_paは最低3あるが、例えば2列目の変数の候補は[0]しかない。(1は順序を一意に保つために使う)
        # そのためlen(candidate)との大小関係もチェックが要る。
        pa = np.random.choice(candidates, size=min(num_pa, len(candidates)), replace=False)

        # the corresponding edge weights are set to +-1/5
        dag[v, pa] = np.random.choice([-1/5, 1/5], size=len(pa))

    # The n error terms for variable v are generated by selecting σv ∼ unif(.8, 1) drawing εvi ∼ σvunif(−√3,√3).
    v = np.random.uniform(0.8, 1, size=n_nodes)
    errors = v * np.random.uniform(-np.sqrt(3), np.sqrt(3), size=(sample_size, n_nodes))

    return dag, errors

dag, errors = make_dag_and_errors(30, 100)
X = (np.linalg.pinv(np.eye(len(dag)) - dag) @ errors.T).T
    
high_d_model = HighDimDirectLiNGAM()
%prun high_d_model.fit(X)

None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None


KeyboardInterrupt: 